# Module 6: Training a Diffusion Model

**Learning objectives:** Data loading, DDPM Algorithm 1, timestep sampling, noise prediction, loss computation, EMA, LR schedules, training pitfalls.

**Estimated time:** 3-4 hours (including training time)

**Key Papers:**
- DDPM -- Ho et al. 2020. [arxiv.org/abs/2006.11239](https://arxiv.org/abs/2006.11239) -- Algorithm 1
- Improved DDPM -- Nichol & Dhariwal 2021. [arxiv.org/abs/2102.09672](https://arxiv.org/abs/2102.09672)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import math, os, copy
from typing import Dict, Tuple, Optional, List

torch.manual_seed(42)
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

---
## UNet Architecture

We use the UNet built in **Module 4** and exported to `utils/unet.py`. This ensures architectural consistency
across modules — the same model that was designed and tested in Module 4 is the one we train here,
and the checkpoint will load cleanly in Module 7 for sampling.

The UNet takes `(x, t, class_label=None)` where:
- `x`: noisy image `(B, C, H, W)`
- `t`: integer timesteps `(B,)`
- `class_label`: optional integer class labels `(B,)` for conditional generation

Key architectural choices (see Module 4 for full details):

| Component | Details |
|---|---|
| **Activation** | SiLU throughout |
| **Time embedding** | Sinusoidal → 2-layer MLP |
| **Skip connections** | All ResBlock outputs + downsample outputs stored and consumed by decoder |
| **Attention** | Custom spatial self-attention at specified resolutions |
| **Upsampling** | Nearest-neighbor upsample + Conv2d (not transposed conv) |

In [ ]:
# Import the canonical UNet from utils/ (built in Module 4)
import sys
sys.path.insert(0, '.')
from utils.unet import UNet

# Quick shape check
model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
dummy_x = torch.randn(2, 1, 28, 28, device=device)
dummy_t = torch.randint(0, 1000, (2,), device=device)
out = model(dummy_x, dummy_t)
print(f"Input shape:  {dummy_x.shape}")   # (2, 1, 28, 28)
print(f"Output shape: {out.shape}")        # (2, 1, 28, 28)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
del model, dummy_x, dummy_t, out  # free memory

---
## Noise Schedules

We import the noise schedule from `utils/schedule.py` (also built in earlier modules).

| Schedule | Formula | Behavior |
|---|---|---|
| **Linear** | `beta_t` linearly from `beta_start` to `beta_end` | Simple, but too aggressive at high `t` |
| **Cosine** | `alpha_bar_t = f(t)/f(0)` where `f(t) = cos((t/T + s)/(1+s) * pi/2)^2` | Smoother, better for small images |

Both return a dictionary of pre-computed tensors that we index by timestep during training.

In [ ]:
from utils.schedule import linear_schedule, cosine_schedule, get_schedule

# Compare schedules visually
sched_lin = linear_schedule(1000)
sched_cos = cosine_schedule(1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ts = np.arange(1000)
axes[0].plot(ts, sched_lin['alphas_cumprod'].numpy(), label='Linear')
axes[0].plot(ts, sched_cos['alphas_cumprod'].numpy(), label='Cosine')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('alpha_bar_t')
axes[0].set_title('Cumulative Signal Retention')
axes[0].legend()

axes[1].plot(ts, sched_lin['betas'].numpy(), label='Linear')
axes[1].plot(ts, sched_cos['betas'].numpy(), label='Cosine')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('beta_t')
axes[1].set_title('Per-Step Noise Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Linear  -- alpha_bar at t=500: {sched_lin['alphas_cumprod'][500]:.4f}")
print(f"Cosine  -- alpha_bar at t=500: {sched_cos['alphas_cumprod'][500]:.4f}")

---
## 6.1 -- Data Loading and Preprocessing

Diffusion models operate on images normalized to **[-1, 1]**. Why?

- The forward process adds Gaussian noise (mean 0), so the data should be centered at 0.
- At `t = T` the noisy image is approximately `N(0, I)`, which lives in the same range as the data when data is in [-1, 1].
- Symmetric range keeps the loss landscape balanced.

We use MNIST at 28x28 for feasible training on CPU/MPS.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(28),
    transforms.ToTensor(),                       # [0, 1]
    transforms.Normalize([0.5], [0.5]),           # [-1, 1]
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True, num_workers=0)


def denormalize(x: torch.Tensor) -> torch.Tensor:
    """Map tensor from [-1, 1] back to [0, 1] for visualization."""
    return (x.clamp(-1, 1) + 1) / 2


# Verify: show a batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")     # (64, 1, 28, 28)
print(f"Value range: [{sample_batch.min():.2f}, {sample_batch.max():.2f}]")

fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i in range(8):
    axes[i].imshow(denormalize(sample_batch[i, 0]).numpy(), cmap='gray')
    axes[i].set_title(str(sample_labels[i].item()), fontsize=9)
    axes[i].axis('off')
plt.suptitle('Sample Training Images (denormalized)', fontsize=11)
plt.tight_layout()
plt.show()

### Exercise 6.1 -- Verify Normalization

Write code that iterates over the first 10 batches and confirms:
1. The minimum value across all batches is approximately -1.0
2. The maximum value across all batches is approximately 1.0
3. The mean is approximately 0.0

Print the global min, max, and mean.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

global_min = float('inf')
global_max = float('-inf')
running_sum = 0.0
running_count = 0

for i, (batch, _) in enumerate(train_loader):
    if i >= 10:
        break
    global_min = min(global_min, batch.min().item())
    global_max = max(global_max, batch.max().item())
    running_sum += batch.sum().item()
    running_count += batch.numel()

global_mean = running_sum / running_count

print(f"Global min:  {global_min:.4f}  (expect ~ -1.0)")
print(f"Global max:  {global_max:.4f}  (expect ~  1.0)")
print(f"Global mean: {global_mean:.4f}  (expect ~  0.0)")

assert global_min >= -1.01, "Min should be >= -1"
assert global_max <= 1.01,  "Max should be <= 1"

---
## 6.2 -- The Training Algorithm (DDPM Algorithm 1)

The entire DDPM training loop fits in a few lines. From Ho et al. 2020, Algorithm 1:

```
repeat:
  x_0 ~ q(x_0)                                          # sample a clean image
  t   ~ Uniform({1, ..., T})                             # random timestep
  eps ~ N(0, I)                                          # sample noise
  x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps   # forward diffusion
  loss = || eps - eps_theta(x_t, t) ||^2                 # noise prediction MSE
  gradient step on loss
until converged
```

Each term has a clear role:

| Symbol | Meaning |
|---|---|
| `x_0` | Clean training image from dataset |
| `t` | Randomly chosen noise level (higher = noisier) |
| `eps` | The ground-truth noise we added |
| `x_t` | The noisy image at timestep `t` |
| `eps_theta(x_t, t)` | Our UNet's prediction of what noise was added |
| `loss` | How wrong the prediction is -- simple MSE |

In [ ]:
def q_sample(
    x_0: torch.Tensor,
    t: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    noise: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Forward diffusion: sample x_t given x_0 and t.
    
    Args:
        x_0: clean images, (B, C, H, W)
        t: timesteps, (B,)
        schedule: dict of precomputed schedule tensors
        noise: optional pre-generated noise
    Returns:
        (x_t, noise) -- noisy images and the noise that was added
    """
    if noise is None:
        noise = torch.randn_like(x_0)  # (B, C, H, W)

    sqrt_alpha_bar = schedule['sqrt_alphas_cumprod'][t]            # (B,)
    sqrt_one_minus_alpha_bar = schedule['sqrt_one_minus_alphas_cumprod'][t]  # (B,)

    # Reshape for broadcasting: (B,) -> (B, 1, 1, 1)
    sqrt_alpha_bar = sqrt_alpha_bar[:, None, None, None]
    sqrt_one_minus_alpha_bar = sqrt_one_minus_alpha_bar[:, None, None, None]

    x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise  # (B, C, H, W)
    return x_t, noise


def train_step(
    model: nn.Module,
    x_0: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    schedule: Dict[str, torch.Tensor],
    device: torch.device,
    num_timesteps: int = 1000,
) -> float:
    """One training step of DDPM Algorithm 1.
    
    Returns the scalar loss value.
    """
    model.train()
    x_0 = x_0.to(device)                                          # (B, C, H, W)
    batch_size = x_0.shape[0]

    # 1. Sample random timesteps
    t = torch.randint(0, num_timesteps, (batch_size,), device=device)  # (B,)

    # 2. Forward diffusion
    # Move schedule tensors to correct device for indexing
    sched_device = {k: v.to(device) for k, v in schedule.items()}
    x_t, noise = q_sample(x_0, t, sched_device)                   # (B, C, H, W) each

    # 3. Predict noise
    noise_pred = model(x_t, t)                                     # (B, C, H, W)

    # 4. Loss
    loss = F.mse_loss(noise_pred, noise)

    # 5. Backprop
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item()


print("train_step defined.")

### Exercise 6.2 -- Run One Training Step

Instantiate a fresh UNet and optimizer (Adam, lr=2e-4). Run a single training step on one batch from the DataLoader. Verify:
1. The loss is a finite scalar.
2. The loss is approximately 1.0 (an untrained model predicting random noise on standard-normal targets has expected MSE near 1).

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)
test_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
test_optimizer = torch.optim.Adam(test_model.parameters(), lr=2e-4)
schedule = cosine_schedule(1000)

test_batch, _ = next(iter(train_loader))
loss_val = train_step(test_model, test_batch, test_optimizer, schedule, device)

print(f"Single-step loss: {loss_val:.4f}")
assert math.isfinite(loss_val), "Loss is not finite!"
assert 0.3 < loss_val < 3.0, f"Loss {loss_val} is unexpectedly far from 1.0"
print("Passed: loss is finite and in expected range.")

del test_model, test_optimizer  # free memory

---
## 6.3 -- Random Timestep Sampling

**Uniform sampling** draws `t ~ Uniform({0, ..., T-1})` each step. This is what DDPM uses and works well in practice.

**Importance sampling** is an alternative: some timesteps contribute more to the loss than others. If we knew the per-timestep loss `L(t)`, we could sample `t` proportionally to `L(t)` and reweight the gradient. Nichol & Dhariwal (2021) explored this but found uniform sampling with enough steps works comparably.

Below we visualize how loss varies by timestep for an untrained model.

In [ ]:
torch.manual_seed(42)
probe_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
schedule = cosine_schedule(1000)
sched_device = {k: v.to(device) for k, v in schedule.items()}

probe_batch, _ = next(iter(train_loader))  # (64, 1, 28, 28)
probe_batch = probe_batch.to(device)

# Measure loss at specific timesteps
timestep_values = list(range(0, 1000, 50))
losses_by_t = []

probe_model.eval()
with torch.no_grad():
    for t_val in timestep_values:
        t_tensor = torch.full((probe_batch.shape[0],), t_val, device=device, dtype=torch.long)
        x_t, noise = q_sample(probe_batch, t_tensor, sched_device)
        noise_pred = probe_model(x_t, t_tensor)
        loss = F.mse_loss(noise_pred, noise).item()
        losses_by_t.append(loss)

plt.figure(figsize=(8, 4))
plt.plot(timestep_values, losses_by_t, 'o-', markersize=3)
plt.xlabel('Timestep t')
plt.ylabel('MSE Loss')
plt.title('Loss vs. Timestep (untrained model, cosine schedule)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss at t=0:   {losses_by_t[0]:.4f}  (almost clean -- easy)")
print(f"Loss at t=500: {losses_by_t[10]:.4f}")
print(f"Loss at t=950: {losses_by_t[-1]:.4f}  (nearly pure noise -- hard)")

del probe_model

### Exercise 6.3 -- Implement Importance-Weighted Timestep Sampling

Implement a function `sample_timesteps_importance` that:
1. Maintains a running average of per-timestep losses in `T` bins.
2. Samples timesteps proportionally to `loss_per_bin + epsilon` (to avoid zero-probability bins).
3. Returns both the sampled timesteps and importance weights `w_t = 1 / (T * p_t)` for unbiased gradient estimation.

You do not need to integrate this into training -- just implement the sampling logic.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class ImportanceTimestepSampler:
    """Importance-weighted timestep sampling based on running loss estimates."""

    def __init__(self, num_timesteps: int = 1000, epsilon: float = 1e-3) -> None:
        self.num_timesteps = num_timesteps
        self.epsilon = epsilon
        # Running average of losses per timestep bin
        self.loss_bins = torch.ones(num_timesteps)  # initialize uniform
        self.counts = torch.zeros(num_timesteps)

    def update(self, timesteps: torch.Tensor, losses: torch.Tensor) -> None:
        """Update running loss estimates with observed (timestep, loss) pairs."""
        for t_val, loss_val in zip(timesteps.cpu(), losses.cpu()):
            t_idx = t_val.long().item()
            self.counts[t_idx] += 1
            # Exponential moving average
            alpha = 1.0 / self.counts[t_idx].item()
            self.loss_bins[t_idx] = (1 - alpha) * self.loss_bins[t_idx] + alpha * loss_val.item()

    def sample(self, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample timesteps proportional to loss + epsilon.
        
        Returns:
            timesteps: (B,) sampled timestep indices
            weights: (B,) importance weights for unbiased gradients
        """
        probs = self.loss_bins + self.epsilon          # (T,)
        probs = probs / probs.sum()                    # normalize
        timesteps = torch.multinomial(probs, batch_size, replacement=True)  # (B,)
        # Importance weight: w_t = 1 / (T * p_t)
        weights = 1.0 / (self.num_timesteps * probs[timesteps])  # (B,)
        return timesteps, weights


# Demonstrate
sampler = ImportanceTimestepSampler(num_timesteps=1000)

# Simulate: pretend high timesteps have higher loss
fake_ts = torch.arange(0, 1000, 10)
fake_losses = torch.linspace(0.5, 2.0, len(fake_ts))
sampler.update(fake_ts, fake_losses)

sampled_ts, sampled_weights = sampler.sample(10000)

plt.figure(figsize=(8, 3))
plt.hist(sampled_ts.numpy(), bins=50, density=True, alpha=0.7)
plt.xlabel('Timestep')
plt.ylabel('Sampling density')
plt.title('Importance sampling: higher-loss timesteps sampled more often')
plt.tight_layout()
plt.show()

print(f"Mean weight: {sampled_weights.mean():.4f} (should be ~1.0 for unbiased estimation)")

---
## 6.4 -- Noise Prediction Forward Pass

Let us walk through the forward pass step by step to build intuition for what happens inside `train_step`.

In [ ]:
torch.manual_seed(42)
demo_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
schedule = cosine_schedule(1000)
sched_device = {k: v.to(device) for k, v in schedule.items()}

# Step 1: Get a clean image batch
x_0, _ = next(iter(train_loader))  # (64, 1, 28, 28)
x_0 = x_0.to(device)
print(f"1. Clean images x_0:          {x_0.shape}")

# Step 2: Sample random timesteps
t = torch.randint(0, 1000, (x_0.shape[0],), device=device)  # (64,)
print(f"2. Timesteps t:               {t.shape}, range [{t.min()}, {t.max()}]")

# Step 3: Forward diffusion -- add noise
x_t, noise = q_sample(x_0, t, sched_device)  # (64, 1, 28, 28) each
print(f"3. Noisy images x_t:          {x_t.shape}")
print(f"   Ground-truth noise eps:    {noise.shape}")

# Step 4: Model prediction
demo_model.eval()
with torch.no_grad():
    noise_pred = demo_model(x_t, t)  # (64, 1, 28, 28)
print(f"4. Predicted noise eps_theta: {noise_pred.shape}")

# Step 5: Compute loss
loss = F.mse_loss(noise_pred, noise)
print(f"5. MSE loss:                  {loss.item():.4f}")

# Visualize one example
idx = 0
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
titles = [f'x_0 (clean)', f'x_t (t={t[idx].item()})', 'True noise', 'Predicted noise']
tensors = [x_0[idx, 0], x_t[idx, 0], noise[idx, 0], noise_pred[idx, 0]]
for ax, title, tensor in zip(axes, titles, tensors):
    ax.imshow(tensor.cpu().numpy(), cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.suptitle('Forward Pass Walkthrough', fontsize=12)
plt.tight_layout()
plt.show()

del demo_model

### Exercise 6.4 -- Shape Verification

Write a function `verify_forward_pass` that:
1. Creates a UNet with `base_channels=32` (smaller, for speed).
2. Generates a random batch of shape `(4, 1, 28, 28)` and random timesteps.
3. Asserts the output shape matches the input shape.
4. Asserts the output dtype matches the input dtype.
5. Asserts all output values are finite.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def verify_forward_pass() -> None:
    """Verify UNet forward pass shapes, dtypes, and finiteness."""
    torch.manual_seed(42)
    test_model = UNet(image_channels=1, base_channels=32, channel_mults=(1, 2, 4)).to(device)
    test_model.eval()

    x = torch.randn(4, 1, 28, 28, device=device)         # (4, 1, 28, 28)
    t = torch.randint(0, 1000, (4,), device=device)       # (4,)

    with torch.no_grad():
        out = test_model(x, t)                             # (4, 1, 28, 28)

    assert out.shape == x.shape, f"Shape mismatch: {out.shape} != {x.shape}"
    assert out.dtype == x.dtype, f"Dtype mismatch: {out.dtype} != {x.dtype}"
    assert torch.isfinite(out).all(), "Output contains non-finite values"

    print(f"Input shape:  {x.shape}, dtype: {x.dtype}")
    print(f"Output shape: {out.shape}, dtype: {out.dtype}")
    print(f"All finite:   {torch.isfinite(out).all().item()}")
    print("All assertions passed.")

    del test_model

verify_forward_pass()

---
## 6.5 -- Loss Computation and Backpropagation

The loss in DDPM is simple MSE between the true noise and the predicted noise:

```
L_simple = E_{t, x_0, eps} [ || eps - eps_theta(x_t, t) ||^2 ]
```

Key details for stable training:

| Technique | Why |
|---|---|
| `F.mse_loss` | Standard L2 loss -- simple and effective for noise prediction |
| Gradient clipping | Prevents gradient explosions; `clip_grad_norm_(params, 1.0)` is standard |
| `optimizer.zero_grad()` | Must clear old gradients before each backward pass |
| Order: zero_grad -> forward -> loss -> backward -> clip -> step | Getting this wrong causes subtle bugs |

In [ ]:
# Demonstrate the full backward pass with gradient monitoring

torch.manual_seed(42)
loss_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
optimizer = torch.optim.Adam(loss_model.parameters(), lr=2e-4)
schedule = cosine_schedule(1000)
sched_device = {k: v.to(device) for k, v in schedule.items()}

x_0, _ = next(iter(train_loader))
x_0 = x_0.to(device)  # (64, 1, 28, 28)

# Step-by-step backward pass
loss_model.train()

# 1. Zero gradients
optimizer.zero_grad()

# 2. Sample timesteps and noise
t = torch.randint(0, 1000, (x_0.shape[0],), device=device)
x_t, noise = q_sample(x_0, t, sched_device)

# 3. Forward pass
noise_pred = loss_model(x_t, t)  # (64, 1, 28, 28)

# 4. Compute loss
loss = F.mse_loss(noise_pred, noise)
print(f"Loss value: {loss.item():.4f}")

# 5. Backward
loss.backward()

# Check gradient norms BEFORE clipping
grad_norm_before = torch.nn.utils.clip_grad_norm_(loss_model.parameters(), max_norm=float('inf'))
print(f"Gradient norm (before clipping): {grad_norm_before:.4f}")

# Re-run backward (need to re-zero and re-compute since clip_grad_norm_ modifies in place)
optimizer.zero_grad()
noise_pred = loss_model(x_t, t)
loss = F.mse_loss(noise_pred, noise)
loss.backward()

# 6. Clip gradients
grad_norm_clipped = torch.nn.utils.clip_grad_norm_(loss_model.parameters(), max_norm=1.0)
print(f"Gradient norm (after clipping):  {min(grad_norm_clipped.item(), 1.0):.4f}")

# 7. Optimizer step
optimizer.step()

print("Backward pass completed successfully.")
del loss_model, optimizer

### Exercise 6.5 -- Compare L1 vs L2 Loss

Run 50 training steps using `F.l1_loss` and 50 steps using `F.mse_loss` on the same data. Plot both loss curves. Which converges faster from a random initialization?

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def run_steps_with_loss_fn(loss_fn, num_steps: int = 50) -> List[float]:
    """Run training steps with a given loss function and return loss history."""
    torch.manual_seed(42)
    model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    sched = cosine_schedule(1000)
    sched_dev = {k: v.to(device) for k, v in sched.items()}
    losses = []
    loader_iter = iter(train_loader)

    model.train()
    for step in range(num_steps):
        try:
            x_0, _ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            x_0, _ = next(loader_iter)

        x_0 = x_0.to(device)
        t = torch.randint(0, 1000, (x_0.shape[0],), device=device)
        x_t, noise = q_sample(x_0, t, sched_dev)
        noise_pred = model(x_t, t)
        loss = loss_fn(noise_pred, noise)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())

    del model, opt
    return losses


l1_losses = run_steps_with_loss_fn(F.l1_loss, 50)
l2_losses = run_steps_with_loss_fn(F.mse_loss, 50)

plt.figure(figsize=(8, 4))
plt.plot(l1_losses, label='L1 loss', alpha=0.8)
plt.plot(l2_losses, label='L2 (MSE) loss', alpha=0.8)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('L1 vs L2 Loss over 50 Training Steps')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final L1 loss: {l1_losses[-1]:.4f}")
print(f"Final L2 loss: {l2_losses[-1]:.4f}")
print("Note: L1 and L2 are on different scales, so compare relative decrease, not absolute values.")

---
## 6.6 -- Exponential Moving Average (EMA)

EMA maintains a shadow copy of model weights that is a smoothed version of the training weights:

```
shadow_param = decay * shadow_param + (1 - decay) * current_param
```

Why EMA matters for diffusion models:

| Aspect | Training weights | EMA weights |
|---|---|---|
| Noise | More noisy (recent gradient updates) | Smoothed over many steps |
| Sample quality | Lower | Higher |
| Typical decay | -- | 0.9999 (very slow averaging) |

At inference time, we swap in the EMA weights for higher-quality samples.

In [ ]:
class EMA:
    """Exponential Moving Average of model parameters.
    
    Usage:
        ema = EMA(model, decay=0.9999)
        # In training loop: ema.update(model)
        # For inference: ema.apply_shadow(model), then model(...), then ema.restore(model)
    """

    def __init__(self, model: nn.Module, decay: float = 0.9999) -> None:
        self.decay = decay
        self.shadow: Dict[str, torch.Tensor] = {}
        self.backup: Dict[str, torch.Tensor] = {}
        # Initialize shadow parameters as copies of current parameters
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model: nn.Module) -> None:
        """Update shadow parameters with current model parameters."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] + (1.0 - self.decay) * param.data
                )

    def apply_shadow(self, model: nn.Module) -> None:
        """Replace model parameters with shadow parameters (for inference)."""
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()

    def restore(self, model: nn.Module) -> None:
        """Restore original model parameters (after inference)."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name].clone()
        self.backup = {}


# Demonstrate EMA behavior
torch.manual_seed(42)
demo = nn.Linear(10, 10)
ema_demo = EMA(demo, decay=0.99)

# Simulate parameter updates
original_weight = demo.weight.data[0, 0].item()
for _ in range(100):
    # Simulate a gradient step (random perturbation)
    demo.weight.data += torch.randn_like(demo.weight.data) * 0.1
    ema_demo.update(demo)

print(f"Original weight[0,0]:  {original_weight:.4f}")
print(f"Current weight[0,0]:   {demo.weight.data[0, 0].item():.4f}  (noisy from updates)")
print(f"EMA shadow[0,0]:       {ema_demo.shadow['weight'][0, 0].item():.4f}  (smoothed)")

del demo, ema_demo

### Exercise 6.6 -- EMA Convergence Test

Create a simple 1D experiment:
1. A parameter starts at 0.0.
2. Each step, set it to a noisy version of target value 5.0 (e.g., `5.0 + randn * 0.5`).
3. Track the raw parameter and EMA parameter over 200 steps.
4. Plot both. The EMA line should be smoother and converge to ~5.0.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)
param = nn.Parameter(torch.tensor(0.0))
shadow = 0.0
decay = 0.99
target = 5.0

raw_values = []
ema_values = []

for step in range(200):
    # Simulate noisy update toward target
    noisy_val = target + torch.randn(1).item() * 0.5
    param.data = torch.tensor(noisy_val)
    # EMA update
    shadow = decay * shadow + (1 - decay) * param.data.item()
    raw_values.append(param.data.item())
    ema_values.append(shadow)

plt.figure(figsize=(10, 4))
plt.plot(raw_values, alpha=0.5, label='Raw parameter (noisy)', linewidth=0.8)
plt.plot(ema_values, label=f'EMA (decay={decay})', linewidth=2)
plt.axhline(y=target, color='r', linestyle='--', label=f'Target = {target}')
plt.xlabel('Step')
plt.ylabel('Value')
plt.title('EMA Smoothing Demonstration')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final raw value: {raw_values[-1]:.4f}")
print(f"Final EMA value: {ema_values[-1]:.4f}")
print(f"EMA is closer to target: {abs(ema_values[-1] - target) < abs(raw_values[-1] - target)}")

---
## 6.7 -- Learning Rate Schedules

A good LR schedule for diffusion training combines:

1. **Linear warmup** (first N steps): ramp LR from 0 to peak. Prevents early instability when gradients are large.
2. **Cosine decay** (remaining steps): smoothly decrease LR to near 0. Allows fine-grained convergence.

```
lr(step) = {
  peak_lr * (step / warmup_steps)                            if step < warmup_steps
  peak_lr * 0.5 * (1 + cos(pi * (step - warmup) / decay))   otherwise
}
```

In [ ]:
def get_warmup_cosine_scheduler(
    optimizer: torch.optim.Optimizer,
    warmup_steps: int,
    total_steps: int,
) -> torch.optim.lr_scheduler.LambdaLR:
    """Warmup + cosine decay learning rate scheduler."""

    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Visualize the schedule
dummy_model = nn.Linear(10, 10)
dummy_opt = torch.optim.Adam(dummy_model.parameters(), lr=2e-4)
total_steps = 5000
warmup_steps = 500
scheduler = get_warmup_cosine_scheduler(dummy_opt, warmup_steps, total_steps)

lrs = []
for step in range(total_steps):
    lrs.append(dummy_opt.param_groups[0]['lr'])
    scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.axvline(x=warmup_steps, color='r', linestyle='--', alpha=0.5, label=f'Warmup ends ({warmup_steps} steps)')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay Schedule')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak LR: {max(lrs):.6f}")
print(f"Final LR: {lrs[-1]:.6f}")

del dummy_model, dummy_opt

### Exercise 6.7 -- Implement a Custom Warmup Schedule

Implement `get_warmup_linear_decay_scheduler` that:
1. Linearly warms up for `warmup_steps`.
2. Then linearly decays to 0 over the remaining steps.

Plot it alongside the cosine version for comparison.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def get_warmup_linear_decay_scheduler(
    optimizer: torch.optim.Optimizer,
    warmup_steps: int,
    total_steps: int,
) -> torch.optim.lr_scheduler.LambdaLR:
    """Warmup + linear decay learning rate scheduler."""

    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 1.0 - progress)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Compare both schedules
total_steps = 5000
warmup_steps = 500

cosine_lrs = []
linear_lrs = []

for name, sched_fn, lr_list in [
    ('Cosine', get_warmup_cosine_scheduler, cosine_lrs),
    ('Linear', get_warmup_linear_decay_scheduler, linear_lrs),
]:
    m = nn.Linear(10, 10)
    o = torch.optim.Adam(m.parameters(), lr=2e-4)
    s = sched_fn(o, warmup_steps, total_steps)
    for step in range(total_steps):
        lr_list.append(o.param_groups[0]['lr'])
        s.step()
    del m, o

plt.figure(figsize=(10, 4))
plt.plot(cosine_lrs, label='Warmup + Cosine Decay')
plt.plot(linear_lrs, label='Warmup + Linear Decay')
plt.axvline(x=warmup_steps, color='r', linestyle='--', alpha=0.5, label='Warmup ends')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('LR Schedule Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6.8 -- Common Training Pitfalls

| Symptom | Likely Cause | Fix |
|---|---|---|
| **Loss spikes** | Learning rate too high, or a bad batch | Reduce LR, add gradient clipping, increase warmup |
| **Loss plateaus early** | Model too small, LR too low, or data issue | Increase model capacity (`base_channels`), tune LR |
| **NaN loss** | Numerical overflow (often in attention or norm layers) | Check for division by zero in schedule, use `float32`, add `eps` to denominators |
| **Mode collapse** | Model only generates one or a few modes | Train longer, check data diversity, verify no label leakage |
| **Blurry samples** | Undertrained, or loss does not weight fine details enough | Train longer, try L1 loss or perceptual loss in addition to MSE |
| **Slow convergence** | Batch size too small, poor schedule choice | Increase batch size, switch to cosine schedule, use EMA |
| **Memory errors** | Model or batch too large for device | Reduce `base_channels`, reduce batch size, use gradient accumulation |

In [ ]:
# Demonstration: what happens with a bad learning rate

torch.manual_seed(42)
schedule = cosine_schedule(1000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
learning_rates = [1e-2, 2e-4, 1e-6]
labels = ['LR=1e-2 (too high)', 'LR=2e-4 (good)', 'LR=1e-6 (too low)']

for ax, lr, label in zip(axes, learning_rates, labels):
    torch.manual_seed(42)
    m = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)
    o = torch.optim.Adam(m.parameters(), lr=lr)
    losses = []
    loader_iter = iter(train_loader)

    for step in range(30):
        try:
            batch, _ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch, _ = next(loader_iter)
        loss_val = train_step(m, batch, o, schedule, device)
        losses.append(loss_val)

    ax.plot(losses)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    del m, o

plt.suptitle('Effect of Learning Rate on Training', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6.9 -- Monitoring Training

Loss curves alone are insufficient for evaluating diffusion models. You must also inspect generated samples periodically:

- **Loss curve**: should decrease smoothly. A flat or rising curve means something is wrong.
- **Sample grids**: generate a small grid of images every N steps. Early samples will be noise; as training progresses, structure should emerge.

Below we implement a simple sampling function for monitoring (simplified DDPM sampling, covered in depth in Module 7).

In [ ]:
@torch.no_grad()
def sample_images(
    model: nn.Module,
    schedule: Dict[str, torch.Tensor],
    num_samples: int = 16,
    image_shape: Tuple[int, ...] = (1, 28, 28),
    num_timesteps: int = 1000,
    device: torch.device = device,
) -> torch.Tensor:
    """Generate samples using DDPM reverse process (simplified).
    
    Returns: (num_samples, C, H, W) tensor in [-1, 1].
    """
    model.eval()
    sched = {k: v.to(device) for k, v in schedule.items()}

    # Start from pure noise
    x = torch.randn(num_samples, *image_shape, device=device)  # (N, C, H, W)

    for t_val in reversed(range(num_timesteps)):
        t_tensor = torch.full((num_samples,), t_val, device=device, dtype=torch.long)

        # Predict noise
        noise_pred = model(x, t_tensor)  # (N, C, H, W)

        # DDPM reverse step
        beta_t = sched['betas'][t_val]
        sqrt_recip_alpha_t = sched['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_alpha_bar_t = sched['sqrt_one_minus_alphas_cumprod'][t_val]

        # Predicted mean
        pred_mean = sqrt_recip_alpha_t * (x - beta_t / sqrt_one_minus_alpha_bar_t * noise_pred)

        if t_val > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(sched['posterior_variance'][t_val])
            x = pred_mean + sigma_t * noise
        else:
            x = pred_mean

    model.train()
    return x  # (N, C, H, W)


def show_sample_grid(
    samples: torch.Tensor,
    nrow: int = 4,
    title: str = 'Generated Samples',
) -> None:
    """Display a grid of generated samples."""
    n = samples.shape[0]
    ncol = nrow
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.5, nrow * 1.5))
    for i in range(nrow):
        for j in range(ncol):
            idx = i * ncol + j
            if idx < n:
                axes[i, j].imshow(denormalize(samples[idx, 0]).cpu().numpy(), cmap='gray')
            axes[i, j].axis('off')
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


print("Sampling and visualization functions defined.")

---
## Capstone -- Full Training on MNIST

We now assemble everything into a complete training loop:

| Setting | Value |
|---|---|
| Dataset | MNIST 28x28, normalized to [-1, 1] |
| Schedule | Cosine, T=1000 |
| Optimizer | Adam, peak LR=2e-4 |
| Gradient clipping | max_norm=1.0 |
| EMA decay | 0.9999 |
| LR schedule | 500-step warmup + cosine decay |
| Training steps | 5000 |
| Sample generation | Every 1000 steps |
| Checkpoint | Saved at end to `checkpoints/ddpm_mnist.pt` |

In [ ]:
# ---- Configuration ----
NUM_TIMESTEPS = 1000
TOTAL_STEPS = 5000
WARMUP_STEPS = 500
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
EMA_DECAY = 0.9999
GRAD_CLIP = 1.0
SAMPLE_EVERY = 1000
NUM_SAMPLES = 16

# ---- Setup ----
torch.manual_seed(42)

model = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_res_blocks=2,
    attention_resolutions=(7,),
    dropout=0.0,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = get_warmup_cosine_scheduler(optimizer, WARMUP_STEPS, TOTAL_STEPS)
schedule = cosine_schedule(NUM_TIMESTEPS)
sched_device = {k: v.to(device) for k, v in schedule.items()}
ema = EMA(model, decay=EMA_DECAY)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Device: {device}")
print(f"Training for {TOTAL_STEPS} steps...")
print()

In [ ]:
# ---- Training Loop ----
loss_history = []
lr_history = []
loader_iter = iter(train_loader)

pbar = tqdm(range(TOTAL_STEPS), desc='Training')
for step in pbar:
    # Get batch (cycle through dataset)
    try:
        x_0, _ = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)
        x_0, _ = next(loader_iter)

    x_0 = x_0.to(device)  # (B, 1, 28, 28)
    batch_size = x_0.shape[0]

    # --- DDPM Algorithm 1 ---
    model.train()
    t = torch.randint(0, NUM_TIMESTEPS, (batch_size,), device=device)  # (B,)
    noise = torch.randn_like(x_0)  # (B, 1, 28, 28)

    sqrt_alpha_bar = sched_device['sqrt_alphas_cumprod'][t][:, None, None, None]  # (B, 1, 1, 1)
    sqrt_one_minus = sched_device['sqrt_one_minus_alphas_cumprod'][t][:, None, None, None]  # (B, 1, 1, 1)
    x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus * noise  # (B, 1, 28, 28)

    noise_pred = model(x_t, t)  # (B, 1, 28, 28)
    loss = F.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()
    ema.update(model)

    loss_val = loss.item()
    loss_history.append(loss_val)
    lr_history.append(optimizer.param_groups[0]['lr'])

    # Update progress bar
    if step % 50 == 0:
        avg_loss = np.mean(loss_history[-50:]) if len(loss_history) >= 50 else np.mean(loss_history)
        pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'lr': f'{lr_history[-1]:.2e}'})

    # Generate samples periodically
    if (step + 1) % SAMPLE_EVERY == 0:
        print(f"\nStep {step + 1}: generating samples with EMA weights...")
        ema.apply_shadow(model)
        samples = sample_images(model, schedule, num_samples=NUM_SAMPLES, device=device)
        show_sample_grid(samples, nrow=4, title=f'Samples at Step {step + 1}')
        ema.restore(model)

print(f"\nTraining complete. Final avg loss: {np.mean(loss_history[-100:]):.4f}")

In [ ]:
# ---- Plot Training Curves ----

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve (smoothed)
window = 100
if len(loss_history) >= window:
    smoothed = np.convolve(loss_history, np.ones(window) / window, mode='valid')
    axes[0].plot(smoothed, linewidth=0.8)
else:
    axes[0].plot(loss_history, linewidth=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE Loss (smoothed)')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# LR curve
axes[1].plot(lr_history, linewidth=0.8)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Final Samples with EMA ----

print("Generating final samples using EMA weights...")
ema.apply_shadow(model)
final_samples = sample_images(model, schedule, num_samples=16, device=device)
show_sample_grid(final_samples, nrow=4, title='Final Samples (EMA weights)')
ema.restore(model)

In [ ]:
# ---- Save Checkpoint ----

os.makedirs('checkpoints', exist_ok=True)
checkpoint_path = 'checkpoints/ddpm_mnist.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'ema_shadow': ema.shadow,
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'image_channels': 1,
        'base_channels': 64,
        'channel_mults': (1, 2, 4),
        'num_res_blocks': 2,
        'attention_resolutions': (7,),
        'num_timesteps': NUM_TIMESTEPS,
    },
    'training_info': {
        'total_steps': TOTAL_STEPS,
        'final_loss': np.mean(loss_history[-100:]),
        'ema_decay': EMA_DECAY,
    },
    'loss_history': loss_history,
}, checkpoint_path)

file_size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"File size: {file_size_mb:.1f} MB")

# Verify checkpoint loads correctly
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"Config: {checkpoint['config']}")
print(f"Final loss: {checkpoint['training_info']['final_loss']:.4f}")

---
## Summary

This module covered the complete DDPM training pipeline:

| Section | Key Takeaway |
|---|---|
| 6.1 Data Loading | Normalize images to [-1, 1] to match Gaussian noise assumptions |
| 6.2 Algorithm 1 | The core loop is simple: sample image, sample timestep, add noise, predict noise, MSE loss |
| 6.3 Timestep Sampling | Uniform sampling works well; importance sampling is a refinement |
| 6.4 Forward Pass | UNet takes `(x_t, t)` and outputs noise prediction with the same shape |
| 6.5 Loss and Backprop | MSE loss + gradient clipping + correct zero_grad/backward/step ordering |
| 6.6 EMA | Smoothed weights produce better samples; use decay=0.9999 |
| 6.7 LR Schedules | Warmup prevents early instability; cosine decay enables fine convergence |
| 6.8 Pitfalls | Know the common failure modes and their fixes |
| 6.9 Monitoring | Always inspect generated samples -- loss alone is not sufficient |

The saved checkpoint at `checkpoints/ddpm_mnist.pt` can be loaded in Module 7 for sampling experiments and Module 8 for advanced techniques.